In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings('ignore')

In [ ]:
dev_original = pd.read_csv("../data/rendimiento_estudiantes_dev.csv")

EJERCICIO 1

In [ ]:
dev_original.sample(5)

In [ ]:
dev_original.info()

In [ ]:
dev_original.describe()

Primer Analisis:

- Rendimiento: Target --> se convierte a numero (multiclase)

- Aparicion de variables NaNs en: horas_estudio, nota_previa, horas_sueno, participacion y nivel socio_economico --> Solucion: reemplazar por el mean de su columna (utilizando el del train_set para no generar data_leakage) --> La cantidad de NaNs se puede ver comparando el total de muestras (5058) y el count de cada var.

- Variables categoricas: escuela --> hacemos one-hot encoding, semestre --> asigno numero entero respetando orden cronologico

- Acceso_internet es una variable normalizada.

- Outliers a analizas: nota_previa = 100 (imposible porque va de 0 a 10), asistencia = 0 (significaria que nunca fue al colegio) --> Hay que analizar mas en profundidad para entender estos outliers y tomar una desicion. Son valores raros / imposibles

- Ademas se normalizara las variables continuas: horas_estudio, asistencia, nota_previa, horas_sueno, participacion, horas_extracurricular, distancia_escuela_km, nivel_socioeconomico, tamano_clase.

Histograma de cantidad de NaNs por variable:

In [ ]:
columnas = []
porcentajes = []
for col in dev_original.columns:
	columna = dev_original[col]
	total_NaNs_en_columna = columna.isna().sum() #sumo cantidad de NaNs 
	porcentaje_NaNs_en_columna = (total_NaNs_en_columna/dev_original.shape[0]) * 100

	columnas.append(col)
	porcentajes.append(porcentaje_NaNs_en_columna)

plt.figure()
plt.bar(columnas, porcentajes)
plt.xticks(rotation=45)
plt.ylabel("Porcentaje de NaNs")
plt.title("Porcentaje de valores faltantes por columna")
plt.tight_layout()
plt.show()

Como se menciono antes, para las variables que contienen NaNs, se reemplazaran por sus medias.

Uso histograma de Rendimiento (multiclase) para ver si el dataset esta balanceado o no:

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(dev_original['rendimiento'], bins=30, edgecolor='black', alpha=0.7)

plt.title("Distribución de rendimiento")
plt.xlabel("rendimiento")
plt.ylabel("Frecuencia")

plt.grid(alpha=0.3)
plt.show()

Ahora analizo histograma de rendimiento binario 

In [ ]:
dev_original['rendimiento_binario'] = dev_original['rendimiento'].apply(
    lambda x: 0 if x == 'Insuficiente' else 1
)

sns.countplot(x='rendimiento_binario', data=dev_original)
plt.title('Distribución de rendimiento binario')
plt.show()

Se puede ver que el dataset esta bastante balanceado tomando en cuenta el rendimiento como multiclase pero desbalanceado para rendimiento binario (hay muchos mas aprobados que desaprobados)

Histogramas de las demas variables

In [ ]:
columnas_dev_original = dev_original.drop('rendimiento', axis=1).columns

for col in columnas_dev_original:
    plt.figure(figsize=(6,4))
    plt.hist(dev_original[col], bins=30, edgecolor='black', alpha=0.7)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.grid(alpha=0.3)
    plt.show()

Analisis Histogramas (analizo las variables con valores extraños)

asistencia:
- Distribucion uniforme, mayormente concentrado en asistencias altas
- 100 muestras en 0, deben ser eliminados porque no son valores posibles y componen un 2% del dataset (100 alumnos no es significativo)

nota_previa:
- Valores concentrados entre 0 - 10 (correcto)
- Outliers con valores mayores a 10, lo cual no tiene sentido. Se debe analizar cuantos son y que hacer con estos --> si son pocos se eliminaran.

horas_extrcurriculares:
- Outliers hacia muchas horas lo cual es posible pero no tan comun (> 20/30 horas es mucho). Ademas la mayoria de los valores estan en 0 o 1 lo que indica una posible falta de informacion.

escuela:
- Muy poca frecuencia en casi la mitad de las escuelas. Se puede ver que hay mucha mas frecuencia en las letras en mayuscula y casi no hay valores para las minusculas. Se debe analizar si es por error o porque estan anotadas dos veces las escuelas (se unificaran con .str.upper() si es por esto)


Boxplots

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

num_cols = ['horas_estudio', 'asistencia', 'nota_previa', 'horas_sueno', 
            'participacion', 'horas_extracurricular', 'distancia_escuela_km', 
            'nivel_socioeconomico', 'tamano_clase']

for i, col in enumerate(num_cols):
    sns.boxplot(x='rendimiento', y=col, data=dev_original, ax=axes[i])
    axes[i].set_title(f'Boxplot de {col}')

plt.tight_layout()
plt.show()

- Las variables: horas_estudio, asistencia, participacion y nivel_socioeconomico tienen una tendencia clara; cuanto mayor es el valor, mejor es el rendimiento del alumno --> Por ende son variables importantes para nuestro modelo

- La variable nota_previa parece tener muchos outliers --> hay valores por encima de 10 lo cual no puede ser correcto por lo que explicamos antes, se confirma que debe tenerse en cuenta fuertemente.

- Para las demas clases se puede ver que no parecen tener mucha informacion util para nuestro modelo pero deben ser igualmente tenidas en cuenta por ahora.

Creo matriz de correlacion para analizar las relaciones entre las variables numericas

In [ ]:
plt.figure(figsize=(12, 8))
num_cols = ['horas_estudio', 'asistencia', 'nota_previa', 'horas_sueno', 
            'participacion', 'horas_extracurricular', 'distancia_escuela_km', 
            'nivel_socioeconomico', 'tamano_clase']

corr_matrix = dev_original[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.show()

Correlacion negativa: 
 - -0.83 --> Fuerte entre nivel_socioeconomico y tamano_clase. Esto tiene mucho sentido porque para las escuelas privadas o de mayor nivel educativo (nivel socioeconomico mas alto), las clases son mas chicas.
- Entre -0.46 y -0.42 --> tamano_clase y participacion (si la clase es grande y hay mucha gente es mas dificil la participacion, tiene sentido), tamano_clase y asistencia (al haber mas gente hay mas chances de que haya mas gente que falte, tiene sentido) y por ultimo tamano_clase y horas_estudio (puede tener sentido pero conectandolo a que como en las clases mas chicas, el nivel educativo es mejor (lo vimos antes), los alumnos sean mas estudiosos)

Correlacion positiva:
- tamano_clase y distancia_escuela_km (0.35) --> no encuentro relacion clara. Podria ser porque las escuelas grandes necesitan mas espacio y estan mas alejadas.
- nivel_socioeconomico y participacion (0.37) --> tiene sentido porque la gente de nivel socioec. mas alto esta en un colegio de alto nivel donde los profesores estan atentos a que los estudiantes esten involucrados en la clase.
- nivel_socioeconomico y asistencia (0.37) --> tiene sentido porque al estar en colegios privados, los padres paguen para que el chico vaya (entonces faltar es "perder dinero") y ademas en estos niveles sociales la educacion es prioridad.
- nivel_socioeconomico y horas_estudio (0.41) --> tiene sentido porque al no tener la necesidad de dedicarle tiempo a otras cosas como trabajo por ejemplo, le pueden dedicar mayor tiempo al estudio.

Sin correlacion:
- horas_extracurricular y horas_sueno --> Son independientes a las demas variables, pueden no aportar tanta informacion al modelo.

Ejercicio 1.2

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

num_cols = ['horas_estudio', 'asistencia', 'nota_previa', 'horas_sueno', 
            'participacion', 'horas_extracurricular', 'distancia_escuela_km', 
            'nivel_socioeconomico', 'tamano_clase']

for i, col in enumerate(num_cols):
    sns.boxplot(x='escuela', y=col, data=dev_original, ax=axes[i])
    axes[i].set_title(f'Boxplot de {col}')

plt.tight_layout()
plt.show()

- Las escuelas difieren entre si en varias features --> mayormente en nivel_socioneconomico, tamano_clase y horas_estudio.
- Los outliers de nota_previa aparecen en todas las escuelas, por ende es un problema generalizado
- Esto va a causar que un modelo entrenado en algunas escuelas puede no generalizar para escuelas nuevas.

Hago correlacion entre h

In [ ]:
dev_temp = dev_original.copy()
dev_temp['escuela'] = dev_temp['escuela'].str.strip().str.upper()

orden = {'Insuficiente': 0, 'Regular': 1, 'Bueno': 2, 'Excelente': 3}
dev_temp['rendimiento_num'] = dev_temp['rendimiento'].map(orden)


num_cols = ['horas_estudio', 'asistencia', 'nota_previa', 'horas_sueno', 
            'participacion', 'horas_extracurricular', 'distancia_escuela_km', 
            'nivel_socioeconomico', 'tamano_clase']

corr_por_escuela = dev_temp.groupby('escuela')[num_cols + ['rendimiento_num']].corr()['rendimiento_num'].unstack('escuela').drop('rendimiento_num')

sns.heatmap(corr_por_escuela, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlación de features con rendimiento por escuela')
plt.tight_layout()
plt.show()

- La correlacion entre las features y rendimiento varia entre las escuelas --> no es consistente
- La clase C es el caso mas extremo porque casi no tiene correlacion con ninguna feature
- Esto muestra que cada escuela tiene su propia "dinamica" complicando la generalizacion del modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(x='escuela', hue='rendimiento', data=dev_temp, ax=axes[0])
axes[0].set_title('Proporción de clases por escuela')

sns.countplot(x='semestre', hue='rendimiento', data=dev_original, ax=axes[1])
axes[1].set_title('Proporción de clases por semestre')

plt.tight_layout()
plt.show()

- El rendimiento por semestre es bastante parecido
- El rendimiento por escuela varia demasiado. Esto causaria un gran problema de prediccion si por ejemplo se entrena el modelo con las escuelas A y C, y se testea en D y G. El modelo prediciria todo excelente cuando es totalmente opuesto. Debido a esto, es indispensable que el split del dataset se haga por escuela y no aleatorio.

1.3 --> Limpieza del Dataset

In [ ]:
dev = dev_original.copy()

Convierto target y semestre en valores numericos

In [ ]:
orden = {'Insuficiente': 0, 'Regular': 1, 'Bueno': 2, 'Excelente': 3}
dev['rendimiento_num'] = dev['rendimiento'].map(orden)

In [ ]:
semestres_ordenados = sorted(dev_original['semestre'].unique(), key=lambda x: (x.split('-')[0], x.split('-')[1]))
orden_semestres = {semestres_ordenados[i]: i for i in range(len(semestres_ordenados))}

dev['semestre_num'] = dev['semestre'].map(orden_semestres)

Unifico las escuelas en todas letras mayuscula (resuelvo repeticion de escuelas)

In [ ]:
dev['escuela'] = dev['escuela'].str.strip().str.upper()

Soluciono problema de valores imposibles en nota_previa

Primero analizo la proporcion de notas mayores a 10:

In [ ]:
valores_notas_prev_mayores_10 = dev['nota_previa'].values > 10
proporcion_valores_imposibles_notas_prev = valores_notas_prev_mayores_10.sum() * 100 / dev['nota_previa'].notna().sum()
print (f'Hay un %{proporcion_valores_imposibles_notas_prev:.2f} de notas imposibles')


Al ser una proporcion tan chica del dataset decido eliminar esas filas. Al ser valores imposibles no puedo asignarles un valor estimado porque no tendria sentido.

Elimino las filas:

In [ ]:
dev = dev[(dev['nota_previa'] <= 10) | (dev['nota_previa'].isna())]


Hago el mismo procedimiento con asistencia

In [ ]:
valores_asistencia_cero = dev['asistencia'].values < 1
proporcion_valores_imposibles_asistencia = valores_asistencia_cero.sum() * 100 / dev['asistencia'].notna().sum()
print (f'Hay un %{proporcion_valores_imposibles_asistencia:.2f} de asistencia nula')

Al ser una proporcion tan chica del dataset, decido eliminar estas filas. Alumnos que no hayan ido al colegio no deberian aportar informacion util para el modelo.

Elimino las filas:

In [ ]:
dev = dev[(dev['asistencia'] >= 1) | (dev['asistencia'].isna())]

Hago One-Hot Encoding para la variable colegio:

In [ ]:
colegios = dev['escuela'].unique()

for cole in colegios:
	dev[f'es_{cole}'] = (dev['escuela'] == cole).astype(int)

In [ ]:
dev.info()

La limpieza fue aplicada correctamente. Luego de hacer el split aplicaremos la normalizacion y agregaremos la media de cada columna para rellenar los valores NaNs.

EJERCICIO 2

In [ ]:
from src.preprocessing import *
from src.data_splitting import *

Splitteo utilizando random

Utilizo dev sin multiclase 

In [ ]:
dev_target_binario = dev.drop('rendimiento_num',axis=1)

In [ ]:
train_random, validation_random = random_split(dev,0.8,rand_state = 42)

Checkeo que los tres tipos de splits se esten aplicando correctamente

In [ ]:

print(train_random.shape, validation_random.shape)
print(train_random['rendimiento_binario'].value_counts(normalize=True))
print(validation_random['rendimiento_binario'].value_counts(normalize=True))

Funcionando correctamente para el random_split

Splitteo utilizando group

In [ ]:
train_group, validation_group = group_split(dev,6,rand_state = 42)

In [ ]:
print(train_group['escuela'].unique())
print(validation_group['escuela'].unique())
# verifico que no haya escuelas en común
print(set(train_group['escuela'].unique()) & set(validation_group['escuela'].unique()))

Funcionando correctamente para el group_split

Splitteo utilizando temporal

In [ ]:
train_temporal, validation_temporal = temporal_split(dev,5)

In [ ]:
print(train_temporal['semestre'].unique())
print(validation_temporal['semestre'].unique())

Funcionando correctamente para temporal_split

Agrego valores para los valores faltantes (NaNs) y normalizo. Lo repito para cada split distinto.

- Elijo columnas a las que agregare valores a las columnas con valores NaN (me fijo las que analice que ya tienen NaNs y agrego las que podrian tener en el test) y para las que voy a normalizar.
- Dejo afuera las binarias (ya estan en escala correcta) y las que ya fueron encodeadeas (representan categorias, no variables continuas con una magnitud real).

In [ ]:
columnas_continuas = ['horas_estudio', 'asistencia', 'nota_previa', 'horas_sueno', 
                      'participacion', 'horas_extracurricular', 'distancia_escuela_km', 
                      'nivel_socioeconomico', 'tamano_clase'] 

In [ ]:
group_key = 'escuela' #variable por la que se agruparan las medias

In [ ]:
reemplazo_NaNs(train_random,validation_random,group_key,columnas_continuas)
train_random_norm, validation_random_norm, media_std_random = normalizar(train_random,validation_random, columnas_continuas)

In [ ]:
reemplazo_NaNs(train_group,validation_group,group_key,columnas_continuas)
train_group_norm, validation_group_norm, media_std_group = normalizar(train_group,validation_group, columnas_continuas)

In [ ]:
reemplazo_NaNs(train_temporal,validation_temporal,group_key,columnas_continuas)
train_temporal_norm, validation_temporal_norm, media_std_temporal = normalizar(train_temporal,validation_temporal, columnas_continuas)

Checkeo que se haya normalizado y se hayan agregado los valores

In [ ]:
train_random_norm.describe().round(2)

In [ ]:
validation_random_norm.describe().round(2)

La normalizacion fue correcta (se ve en las medias y desvios de las que se le aplico) y no hay valores NaN, por ende fueron reemplazados.

Entreno cada modelo por separado

In [ ]:
from src.models import *

In [ ]:
nombres_features = dev_target_binario.drop(['rendimiento_binario','escuela','semestre','rendimiento'],axis=1).columns

In [ ]:
alpha = 0.01
iters = 5000
umbral = 0.5

Modelo R --> Random split

In [ ]:
from src.metrics import *

Tomo clase positiva como 0. Esto es porque el ministerio quiere detectar los estudiantes que estan en riesgo.

In [ ]:
clase_pos = 0

In [ ]:
X_train_R = train_random_norm[nombres_features].values
y_train_R = train_random_norm['rendimiento_binario'].values

Modelo_R = LogisticRegression(X_train_R, y_train_R, nombres_features, L2=0, alpha=alpha, iters=iters)
Modelo_R.fit()

X_validation_R = validation_random_norm[nombres_features].values
y_validation_R = validation_random_norm['rendimiento_binario'].values

prediccion_R = Modelo_R.predecir_proba(X_validation_R)
prediccion_clase_R = Modelo_R.predecir_clase(X_validation_R, umbral)


In [ ]:
mc_R = matriz_confusion(prediccion_clase_R, y_validation_R,clase_positiva=clase_pos)
acc_R = accuracy(prediccion_clase_R, y_validation_R,clase_positiva=clase_pos)
prec_R = precision(prediccion_clase_R, y_validation_R,clase_positiva=clase_pos)
rec_R = recall(prediccion_clase_R, y_validation_R,clase_positiva=clase_pos)
f1_R = F1_score(prediccion_clase_R, y_validation_R,clase_positiva=clase_pos)

print(f'Matriz de confusion:\n{mc_R}')
print(f'Accuracy: {acc_R:.4f}')
print(f'Precision: {prec_R:.4f}')
print(f'Recall: {rec_R:.4f}')
print(f'F1-Score: {f1_R:.4f}')


print("\n Los pesos obtenidos por el modelo son:")
Modelo_R.coefs_con_features()


Curva ROC y AUC-ROC de modelo R

In [ ]:
fpr, tpr, _ = curva_ROC(prediccion_R, y_validation_R, clase_positiva=clase_pos)
auc = AUC_ROC(fpr, tpr)

plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0,1], [0,1], 'k--', label='Random')
plt.xlabel('Tasa FP')
plt.ylabel('Recall (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.show()

Curva PR y AUC-PR de modelo R

In [ ]:
rec, prec, _ = curva_PR(prediccion_R, y_validation_R, clase_positiva=clase_pos)
auc_pr = AUC_PR(rec, prec)

plt.plot(rec, prec, label=f'AUC-PR = {auc_pr:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.legend()
plt.show()

Modelo G --> group split

In [ ]:
X_train_G = train_group_norm[nombres_features].values
y_train_G = train_group_norm['rendimiento_binario'].values

Modelo_G = LogisticRegression(X_train_G,y_train_G, nombres_features,L2=0)
Modelo_G.fit()


X_validation_G = validation_group_norm[nombres_features].values
y_validation_G = validation_group_norm['rendimiento_binario'].values

prediccion_G = Modelo_G.predecir_proba(X_validation_G)
prediccion_clase_G = Modelo_G.predecir_clase(X_validation_G, umbral)

In [ ]:
mc_G = matriz_confusion(prediccion_clase_G, y_validation_G,clase_positiva=clase_pos)
acc_G = accuracy(prediccion_clase_G, y_validation_G,clase_positiva=clase_pos)
prec_G = precision(prediccion_clase_G, y_validation_G,clase_positiva=clase_pos)
rec_G = recall(prediccion_clase_G, y_validation_G,clase_positiva=clase_pos)
f1_G = F1_score(prediccion_clase_G, y_validation_G,clase_positiva=clase_pos)

print(f'Matriz de confusion:\n{mc_G}')
print(f'Accuracy: {acc_G:.4f}')
print(f'Precision: {prec_G:.4f}')
print(f'Recall: {rec_G:.4f}')
print(f'F1-Score: {f1_G:.4f}')

print("\n Los pesos obtenidos por el modelo son:")
Modelo_G.coefs_con_features()

Los pesos que son 0 en las escuelas, muestran justamente que en el train el modelo no conocia esas escuelas. Esto seria un caso mas real porque en el test apareceran esceulas desconocidas (es la idea del modelo, que pueda predecir para escuelas desconocidas).

Curva ROC y AUC-ROC de modelo G

In [ ]:
fpr, tpr, _ = curva_ROC(prediccion_G, y_validation_G, clase_positiva=clase_pos)
auc = AUC_ROC(fpr, tpr)

plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0,1], [0,1], 'k--', label='Random')
plt.xlabel('Tasa FP')
plt.ylabel('Recall (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.show()

Curva PR y AUC-PR de modelo R

In [ ]:
rec, prec, _ = curva_PR(prediccion_G, y_validation_G, clase_positiva=clase_pos)
auc_pr = AUC_PR(rec, prec)

plt.plot(rec, prec, label=f'AUC-PR = {auc_pr:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.legend()
plt.show()

Modelo T --> split temporal

In [ ]:
X_train_T = train_temporal_norm[nombres_features].values
y_train_T = train_temporal_norm['rendimiento_binario'].values

Modelo_T = LogisticRegression(X_train_T, y_train_T, nombres_features, L2=0, alpha=alpha, iters=iters)
Modelo_T.fit()


X_validation_T = validation_temporal_norm[nombres_features].values
y_validation_T = validation_temporal_norm['rendimiento_binario'].values

prediccion_T = Modelo_T.predecir_proba(X_validation_T)
prediccion_clase_T = Modelo_T.predecir_clase(X_validation_T,umbral)

In [ ]:
mc_T = matriz_confusion(prediccion_clase_T, y_validation_T,clase_positiva=clase_pos)
acc_T = accuracy(prediccion_clase_T, y_validation_T,clase_positiva=clase_pos)
prec_T = precision(prediccion_clase_T, y_validation_T,clase_positiva=clase_pos)
rec_T = recall(prediccion_clase_T, y_validation_T,clase_positiva=clase_pos)
f1_T = F1_score(prediccion_clase_T, y_validation_T,clase_positiva=clase_pos)

print(f'Matriz de confusion:\n{mc_T}')
print(f'Accuracy: {acc_T:.4f}')
print(f'Precision: {prec_T:.4f}')
print(f'Recall: {rec_T:.4f}')
print(f'F1-Score: {f1_T:.4f}')

print("\n Los pesos obtenidos por el modelo son:")
Modelo_T.coefs_con_features()

In [ ]:
fpr, tpr, _ = curva_ROC(prediccion_T, y_validation_T, clase_positiva=clase_pos)
auc = AUC_ROC(fpr, tpr)

plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0,1], [0,1], 'k--', label='Random')
plt.xlabel('Tasa FP')
plt.ylabel('Recall (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.show()

Curva PR y AUC-PR de modelo R

In [ ]:
rec, prec, _ = curva_PR(prediccion_T, y_validation_T, clase_positiva=clase_pos)
auc_pr = AUC_PR(rec, prec)

plt.plot(rec, prec, label=f'AUC-PR = {auc_pr:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.legend()
plt.show()

El que peores metricas obtuvo fue el temporal, posiblemente porque aprendio en base a semestres pasados pero los semestres son muy cambiantes, al aparecer uno nuevo, no pudo predecir bien. Se puede ver que el random split es el que da mejores metricas. Esto igualmente es porque las escuelas que estaban en el train son las mismas o algunas de las que estaban en el validation (el modelo aprende conociendo ya las escuelas pero puede fallar con nuevas). Por esto es que para produccion en principio se deberia utilizar el group. Esto se vera reflejado al eliminar la variable "escuela".

Analizando el AUC-ROC de cada uno, se puede ver que todos tienenun AUC alto (mayores a 90%), indicando que tienen buena capacidad de separacion. Igualmente, el umbral 0.5 que se puso fijo en la prueba, da un recall bajo en todos los casos, particularmente en temporal y group. Bajando el umbral podria mejorar la deteccion de estudiantes en riesgo, contemplando que tambien se obtendrian mas falsos positvos. Como el objetivo del ministerio es encontrar estudiantes en riesgo, estaria bien ese procedimiento.

Analizando el AUC-PR de cada uno, se puede ver que el que da mayor valor es el de group. Esto es porque en el split hay mas insuficientes en validation (F y G, lo analizamos antes). Cuando el modelo los detecta bien, el AUC-PR sube, tal que hay mas casos de la calse minoritaria (clase 0) para evaluar.

El AUC-PR es mas informativo que el AUC-ROC en datasets desbalanceados (este es el caso, se puede ver que en el analisis la clase de rendimiento binari esta muy desbalanceada), porque mide "que tan bien el modelo detecta la clase minoritaria", en contraste a la AUC-ROC que puede dar valores altos aunque el modelo falle en esa clase desbalanceada. El PR muestra que aunque el Recall es razonable, se equivoca bastante (hay muchos FP). En cambio el AUC-ROC, se ve bien siempre (en este caso), porque el FPR sigue siendo bajo porque el denominador (FP + TN) es muy grande. 

Ahora obtengo hiperparametros con Cross-Validation

In [ ]:
tipos = ['aleatorio', 'group', 'temporal']
lambdas = np.logspace(-4, 1, 20)
F1s_con_L2 = {tipo: {} for tipo in tipos}  

alfa = 0.01
iteraciones = 2000
umbral = 0.5
clase_pos = 0

dev_c_val = dev.copy()
dev_c_val = dev_c_val.drop('rendimiento_num', axis=1)
y_col = 'rendimiento_binario'

k = 5

mejores_lambdas = {}  

for tipo in tipos:
    if tipo == 'temporal':
        train_t, val_t = temporal_split(dev_c_val, 5)
        reemplazo_NaNs(train_t, val_t, group_key, columnas_continuas)
        train_t_norm, val_t_norm, _ = normalizar(train_t, val_t, columnas_continuas)
        X_train = train_t_norm[nombres_features].values
        y_train = train_t_norm[y_col].values
        X_val = val_t_norm[nombres_features].values
        y_val = val_t_norm[y_col].values
        
        for l in lambdas:
            modelo = LogisticRegression(X_train, y_train, nombres_features, L2=l, alpha=alfa, iters=iteraciones)
            modelo.fit()
            y_pred = modelo.predecir_clase(X_val, umbral)
            F1s_con_L2['temporal'][l] = F1_score(y_pred, y_val, clase_positiva=clase_pos)
    else:
        for l in lambdas:
            c_v_l, _, _, _ = cross_val(dev_c_val, nombres_features, y_col, k, l, alfa, iteraciones, umbral, group_key, columnas_continuas, tipo, clase_pos, False, modelo_clase=LogisticRegression)
            F1s_con_L2[tipo][l] = c_v_l

    F1_values = np.array(list(F1s_con_L2[tipo].values()))
    lambda_values = np.array(list(F1s_con_L2[tipo].keys()))
    mejor_idx = np.nanargmax(F1_values)
    mejor_lambda = lambda_values[mejor_idx]
    valor_mejor_lambda = F1_values[mejor_idx]
    mejores_lambdas[tipo] = mejor_lambda
    print(f'El mejor lambda para {tipo} es {mejor_lambda} con F1 = {valor_mejor_lambda}')

Con el F1 podemos medir que tan bien detecta el modelo la clase que nos importa sin cometer demasiados errores en la prediccion. Si el recall o precision falla. el F1 cae. Es mejor metrica que accuracy (accuracy no depende de que clase sea positiva, siempre favorece a la clase mayoritaria) para clases desbalanceadas porque el accuracy puede ser alto prediciendo siempre la clase mayoritaria.

Grafico F1 vs L2

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, tipo in enumerate(tipos):
    lambda_values = np.array(list(F1s_con_L2[tipo].keys()))
    F1_values = np.array(list(F1s_con_L2[tipo].values()))
    
    axes[idx].plot(lambda_values, F1_values)
    axes[idx].axvline(x=mejores_lambdas[tipo], color='r', linestyle='--', label=f'λ óptimo = {mejores_lambdas[tipo]:.4f}')
    axes[idx].set_xscale('log')
    axes[idx].set_xlabel('λ')
    axes[idx].set_ylabel('F1')
    axes[idx].set_title(f'F1 vs λ ({tipo})')
    axes[idx].legend()

plt.tight_layout()
plt.show()

En los tres casos, el valor optimo de lambda esta en valores muy bajos, esto indica que una regularizacion grande perjudica el rendimiento, ya que, al aumentarlo el F1 disminuye de manera clara.

En el caso del random split, se puede ver un maximo claro para valores chicos de lambda, lo que indica que una leve regularizacion puede ayudar pero con valores mas grandes se genera underfitting.

Para group split sucede algo similar al random pero aun mas extremo. Se ve que ya con aplicar algo de regularizacion el modelo empeora, Lo mismo pasa con temporal solo que se ve que se puede aplicar regularizacion para valores chicos pero mayores que el de group.

Esto demuestra que la regularizacion no es algo critico para este dataset, con valores chicos de lambda se obtiene un buen desempeño.

Calculo distribucion de coeficientes de dos modelos. Los dos usando cross_validation, uno usando Kfolds y el otro GroupKfold

El modelo 1 va a ser con Kfolds y el otro con GroupKfolds

In [ ]:
mejor_l_aleatorio = mejores_lambdas['aleatorio']
mejor_l_group = mejores_lambdas['group']

_,ws1,_,_,_ = cross_val(dev_c_val, nombres_features, y_col, k, mejor_l_aleatorio, alfa, iteraciones, umbral, group_key, columnas_continuas, 'aleatorio',clase_pos, True, LogisticRegression)
_,ws2,_,_,_ = cross_val(dev_c_val, nombres_features, y_col, k, mejor_l_group, alfa, iteraciones, umbral, group_key, columnas_continuas, 'group',clase_pos, True, LogisticRegression)

ws1 = np.array(ws1)  
ws2 = np.array(ws2)

desvios_aleatorio = np.std(ws1, axis=0)
desvios_group = np.std(ws2, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

features_con_bias = ['bias'] + list(nombres_features)

axes[0].bar(features_con_bias, desvios_aleatorio)
axes[0].set_title('Desvío de coeficientes - KFold aleatorio')
axes[0].tick_params(axis='x', rotation=90)

axes[1].bar(features_con_bias, desvios_group)
axes[1].set_title('Desvío de coeficientes - GroupKFold')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

- 1: Las features con mas inestabilidad en GroupKfold son las variables de escuela (E, B, D, mayormente). Sus pesos varian mucho segun que escuela esta en el train.
- 2: Un coeficiente inestable indica que la relacion entre esa feature y el target no es consistente entre diferentes subconjuntos de datos. En este caso, el modelo esta encontrando patrones especificos de ciertas escuelas, en vez de relaciones generales que se amntengan al cambiar el set de entrenamiento.
3: 
- Kfolf --> Cuando en produccion el modelo predice sobre estudiantes de escuelas ya conocidas.
- GroupKfold --> Cuando en produccion el modelo tendra que predecir con escuelas nuevas que no vio en el train, es la estimacion mas realista para este caso.

EJERCICIO 3

Dev sin clase binaria

In [ ]:
dev_target_multiclase = dev.drop('rendimiento_binario',axis=1)

Compruebo funcionamiento de de arbol

In [ ]:
train_g, val_g = group_split(dev_target_multiclase, 6)
reemplazo_NaNs(train_g, val_g, group_key, columnas_continuas)
train_g_norm, val_g_norm, _ = normalizar(train_g, val_g, columnas_continuas)

X_train = train_g_norm[nombres_features].values
y_train = train_g_norm['rendimiento_num'].values

arbol = ArbolDecision(X_train, y_train, max_profundidad=5, min_muestras_hoja=10)
arbol.fit()

X_val = val_g_norm[nombres_features].values
y_pred = arbol.predecir_clase(X_val)
print(y_pred[:10])

Busco mejor configuracion para random forest

In [ ]:
configs = [
    (50, 5, 10),
    (50, 10, 5),
    (100, 10, 5),
]
#utilizo estas combinaciones porque tarda en correr demasiado tiempo. Lo probe dejandolo correr media hora y la mejora fue muy chica (diferencia de 0.05).

mejor_f1_macro = 0
mejores_params = None

for n_arb, prof, min_m in configs:
    train_g, val_g = group_split(dev_target_multiclase, 6)
    reemplazo_NaNs(train_g, val_g, group_key, columnas_continuas)
    train_g_norm, val_g_norm, _ = normalizar(train_g, val_g, columnas_continuas)
    
    X_train = train_g_norm[nombres_features].values
    y_train = train_g_norm['rendimiento_num'].values
    X_val = val_g_norm[nombres_features].values
    y_val = val_g_norm['rendimiento_num'].values
    
    rf = RandomForest(X_train, y_train, nombres_features, n_arboles=n_arb, max_profundidad=prof, min_muestras_hoja=min_m)
    rf.fit()
    y_pred = rf.predecir_clase(X_val)
    
    f1_macro = np.nanmean([F1_score(y_pred, y_val, clase_positiva=c) for c in [0,1,2,3]])
    print(f'n_arboles={n_arb}, prof={prof}, min_m={min_m} -> F1 macro={f1_macro:.4f}')
    
    if f1_macro > mejor_f1_macro:
        mejor_f1_macro = f1_macro
        mejores_params = (n_arb, prof, min_m)

Utilizamos n_arboles = 50, profundidad = 5 y min_m = 10. Fue el que nos dio F1 macro mas alto

F1 macro es el promedio del F1 de todas las clases. Se trata a todas las clases por igual, independiente de cuantos ejemplos tiene cada una.

In [ ]:
modelos_clases = {'LDA': LDA, 'LogMulticlase': LogisticRegressionMulticlase, 'RandomForest': RandomForest}

clases = [0, 1, 2, 3]
nombres_clases = ['Insuficiente', 'Regular', 'Bueno', 'Excelente']
resultados = {}

for nombre_modelo, modelo_clase in modelos_clases.items():
    _, ws, y_preds, y_reales, y_probas = cross_val(dev_target_multiclase, nombres_features, 'rendimiento_num', k, 0, alfa, iteraciones, umbral, group_key, columnas_continuas, 'group', 0, True, modelo_clase)
    resultados[nombre_modelo] = (y_preds, y_reales, y_probas)
    
    print(f'\n--- {nombre_modelo} ---')
    f1s = []
    for c, nombre in zip(clases, nombres_clases):
        p = precision(y_preds, y_reales, clase_positiva=c)
        r = recall(y_preds, y_reales, clase_positiva=c)
        f1 = F1_score(y_preds, y_reales, clase_positiva=c)
        acc = accuracy(y_preds, y_reales, clase_positiva=c)
        f1s.append(f1)
        print(f'{nombre}: Accuracy={acc:.4f}, Precision={p:.4f}, Recall={r:.4f}, F1={f1:.4f}')
    
    print(f'F1 macro = {np.nanmean(f1s)}')

El modelo mas consistente en predecir lo mejor posible cada clase es el LogMulticlase (se puede ver que los valores de F1 de cada clase son visiblemente mayores, al calcular un promedio, ese F1 macro aproximado es mayor que los otros). Por oto lado, como el objetivo del Ministerio es poder predecir los estudiantes en riesgo, se le podria mas mas importancia a los F1 de las clases 0 y 1. De igualmanera, el F1 de esas clases es mayor en el modelo LogMulticlase. Además, se puede analizar que el recall y precision de la clase "Bueno" en Random Forest son extremadamente bajos. Esto implica que el modelo casi nunca predice "Bueno", en cambio clasifica a esos estudiantes como "Excelente",  lo que explica el recall extremadamente alto (0.97) pero Precision baja (0.54)  de esa clase. El modelo tiene un sesgo fuerte hacia predecir "Excelente".

Analizo matrices de confusion

In [ ]:
nombres_clases = ['Insuficiente', 'Regular', 'Bueno', 'Excelente']

for nombre_modelo, (y_preds, y_reales,_) in resultados.items():
    mc = matriz_confusion_multiclase(y_preds, y_reales)
    print(f'\n--- {nombre_modelo} ---')
    
    df_mc = pd.DataFrame(mc, index=nombres_clases, columns=nombres_clases)
    print(df_mc)

Viendo los aciertos de cada clase por cada modelo (diagonales), se puede ver que el mas consistente (como dijimos anteriormente) es el LogMulticlase. RandomForest tiene un sesgo importante hacia Excelente y LDA lo tiene con Bueno y Excelemte. Ademas en LDA y RandomForest no detecta casi "Insuficiente", esto confirma el recall bajo que tienen las clase.

Analizo curvas

Curva ROC con AUC-ROC

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

colores = {'LDA': 'blue', 'LogMulticlase': 'orange', 'RandomForest': 'green'}

for nombre_modelo, modelo_clase in modelos_clases.items():
    _, y_preds, y_reales, y_probas = cross_val(
        dev_target_multiclase, nombres_features, 'rendimiento_num',
        k, 0, alfa, iteraciones, umbral, group_key, columnas_continuas,
        'group', 0, False, modelo_clase
    )
    
    for j, (c, nombre) in enumerate(zip(clases, nombres_clases)):
        proba_clase = y_probas[:, j]
        y_bin = (y_reales == c).astype(int)
        
        fpr, tpr, _ = curva_ROC(proba_clase, y_bin, clase_positiva=1)
        auc = AUC_ROC(fpr, tpr)
        
        axes[j].plot(fpr, tpr, label=f'{nombre_modelo} AUC={auc:.3f}', color=colores[nombre_modelo])
        axes[j].plot([0,1], [0,1], 'k--')
        axes[j].set_title(f'ROC - {nombre}')
        axes[j].set_xlabel('FPR')
        axes[j].set_ylabel('TPR')
        axes[j].legend()

plt.tight_layout()
plt.show()

Al observar las curvas ROC (que muestran la capacidad del modelo de separacion de cada clase sobre el resto), se puede ver que LogMulticlase y LDA (analizando que ambos AUC dan tan alto, en Insuficiente y Regular), son muy buenos para detectar a los estudiantes en riesgo. Los tres modelos tienen mayores complicaciones para la separacion de las clases intermedias pero el peor es RandomForest. Por ultimo, los tres son muy buenos en la separacion de "Excelente"

Curva PR AUC-PR

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colores = {'LDA': 'blue', 'LogMulticlase': 'orange', 'RandomForest': 'green'}

for nombre_modelo, modelo_clase in modelos_clases.items():
    _, y_preds, y_reales, y_probas = cross_val(
        dev_target_multiclase, nombres_features, 'rendimiento_num',
        k, 0, alfa, iteraciones, umbral, group_key, columnas_continuas,
        'group', 0, False, modelo_clase
    )
    
    for j, (c, nombre) in enumerate(zip(clases, nombres_clases)):
        proba_clase = y_probas[:, j]
        y_bin = (y_reales == c).astype(int)
        
        rec, prec, _ = curva_PR(proba_clase, y_bin, clase_positiva=1)
        auc_pr = AUC_PR(rec, prec)
        
        axes[j].plot(rec, prec, label=f'{nombre_modelo} AUC={auc_pr:.3f}', color=colores[nombre_modelo])
        axes[j].set_title(f'PR - {nombre}')
        axes[j].set_xlabel('Recall')
        axes[j].set_ylabel('Precision')
        axes[j].legend()

plt.tight_layout()
plt.show()

Como lo vimos en el ejercicio 2, las curvas PR dan mas informacion que las ROC para datasets desbalanceados.
- Para Insuficiente, LDA (0.644) y LogMulticlase (0.655) son parecidos y mejores que RandomForest (0.448).
- Para Regular y Bueno, LogMulticlase tiene el mejor AUC (0.445 y 0.492), siendo el modelo más consistente para las clases intermedias.
- Para Excelente, LogMulticlase lidera (0.911), seguido por RandomForest (0.850) y LDA (0.836).
- En general, RandomForest tiene el peor AUC-PR en las clases de riesgo (Insuficiente y Regular).


Analizo features importance

In [ ]:
train_g, val_g = group_split(dev_target_multiclase, 6)
reemplazo_NaNs(train_g, val_g, group_key, columnas_continuas)
train_g_norm, val_g_norm, _ = normalizar(train_g, val_g, columnas_continuas)

X_train = train_g_norm[nombres_features].values
y_train = train_g_norm['rendimiento_num'].values

rf = RandomForest(X_train, y_train, nombres_features, n_arboles=50, max_profundidad=15, min_muestras_hoja=3)
rf.fit()

importancias = rf.feature_importances()

plt.figure(figsize=(10, 6))
plt.barh(list(nombres_features), importancias)
plt.title('Importancia de features - Random Forest')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

- Las escuelas son las que menos información aportan, Random Forest usa subconjuntos aleatorios de features, y cuando las escuelas no aportan informacion consistente entre muestras, la importancia promedio baja.
- Las variables más importantes son asistencia y horas_estudio, seguidas por nota_previa y horas_sueno. Siendo consistente con el EDA para las primeras dos, aunque nota_previa y horas_sueno tienen correlaciones bajas con el rendimiento. El RandomForest puede capturar relaciones no lineales (eso explicaria lo de las ultimas dos).

EJERCICIO 4

Necesitamos rebalancear el dataset para poder solucionar el problema de que el modelo tiende a predecir siempre "aprobado" porque es la clase que domina/mayoritaria.

Sin rebalanceo: 

In [ ]:
train_g, val_g = group_split(dev_target_binario, 6, rand_state=42)
reemplazo_NaNs(train_g, val_g, group_key, columnas_continuas)
train_g_norm, val_g_norm, _ = normalizar(train_g, val_g, columnas_continuas)

X_train = train_g_norm[nombres_features].values
y_train = train_g_norm['rendimiento_binario'].values

pi_0 = (y_train == 0).sum() / len(y_train)
pi_1 = (y_train == 1).sum() / len(y_train)
cw = {0: pi_1/pi_0, 1: 1.0}

tecnicas = {
    'Sin rebalanceo':    None,
    'Undersampling':     undersampling,
    'Oversampling dup':  oversampling,
    'SMOTE':             smote,
}

resultados = []

for nombre, reb_fn in tecnicas.items():
    _, y_preds, y_reales, y_probas = cross_val(
        dev_target_binario, nombres_features, 'rendimiento_binario',
        k, mejores_lambdas['group'], alfa, iteraciones, umbral,
        group_key, columnas_continuas, 'group', clase_pos, False, 
        LogisticRegression, rebalanceo_fn=reb_fn
    )
    
    acc = accuracy(y_preds, y_reales, clase_positiva=clase_pos)
    prec = precision(y_preds, y_reales, clase_positiva=clase_pos)
    rec = recall(y_preds, y_reales, clase_positiva=clase_pos)
    f1 = F1_score(y_preds, y_reales, clase_positiva=clase_pos)
    fpr, tpr, _ = curva_ROC(y_probas, y_reales, clase_positiva=clase_pos)
    auc_roc = AUC_ROC(fpr, tpr)
    rec_pr, prec_pr, _ = curva_PR(y_probas, y_reales, clase_positiva=clase_pos)
    auc_pr = AUC_PR(rec_pr, prec_pr)
    
    resultados.append({
        'Modelo': nombre, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1': f1, 'AUC-ROC': auc_roc, 'AUC-PR': auc_pr
    })

#Cost re-weighting lo hago a parte porque es el unico que necesita modificar el modelo 
_, y_preds, y_reales, y_probas = cross_val(
    dev_target_binario, nombres_features, 'rendimiento_binario',
    k, mejores_lambdas['group'], alfa, iteraciones, umbral,
    group_key, columnas_continuas, 'group', clase_pos, False,
    lambda X, y, nf, L2, a, i: LogisticRegression(X, y, nf, L2, a, i, class_weight=cw)
)

acc = accuracy(y_preds, y_reales, clase_positiva=clase_pos)
prec = precision(y_preds, y_reales, clase_positiva=clase_pos)
rec = recall(y_preds, y_reales, clase_positiva=clase_pos)
f1 = F1_score(y_preds, y_reales, clase_positiva=clase_pos)
fpr, tpr, _ = curva_ROC(y_probas, y_reales, clase_positiva=clase_pos)
auc_roc = AUC_ROC(fpr, tpr)
rec_pr, prec_pr, _ = curva_PR(y_probas, y_reales, clase_positiva=clase_pos)
auc_pr = AUC_PR(rec_pr, prec_pr)

resultados.append({
    'Modelo': 'Cost re-weighting', 'Accuracy': acc, 'Precision': prec,
    'Recall': rec, 'F1': f1, 'AUC-ROC': auc_roc, 'AUC-PR': auc_pr
})

pd.DataFrame(resultados).round(4)

Se ve claramente un trade-off entre Recall y Precision entre los modelos con rebalanceo y sin. Al aumentar el Recall se pierde presision. Igualmente como nuestro objetivo es buscar estudiantes en riesgo, es preferible cometer errores pero poder encontrar muchos mas estudiantes en riesgo, los que tienen rebalanceo pierden un 20% de presision pero todos ganan el doble de recall. Ademas, se puede analizar que tanto el F1 como el AUC-ROC y AUC-PR, todos son mayores en los modelos con rebalanceo.

- Analizando la mejor tecnica, tenemos que:
- SMOTE tiene el mejor F1, buena relacion entre Recall y Precision.
- Oversampling tiene el mejor Recall pero menor precision que los demas rebalanceados y el menor F1 de los rebalanceados.
- Cost-re-reighting tiene el mejor AUC-PR y buen F1, tambien tiene el recall alto y buena relacion con precision.

Para este caso claramente es mejor utilizar rebalanceo, aunque se pierda precision, el modelo lograria encontrar muchisimos mas estudiantes en riesgo sin cometer errores imposibles de manejar.

Considerando que el objetivo es maximizar la deteccion de estudiantes en riesgo manteniendo un buen balance, se recomienda SMOTE. Tiene el mejor F1 (0.64), buen Recall (0.92) y genera datos sinteticos nuevos en vez de simplemente duplicar o eliminar muestras.


EJERCICIO 5


Caso Binario:
- Modelo: Regresion Logística con regularización L2 (unico modelo implementado para clasificacion binaria)
- Rebalanceo: SMOTE, porque toeme el mejor F1 (0.64) y un buen recall (0.92).
- Estrategia de splitteo: groupKfold, porque es la estimacion mas realista para preduccion, ya que en produccion el modelo se encontrara con escuelas que nunca vio en train.

Caso Multiclase:
- Modelo: Regresion Logistica Mulitcalse, tal que, tiene el mejor F1 macro y es el mas consistente para todas las clases, especialmente para las clases Insuficiente y Regular, que son las que mas nos interesan detectar.
-Estrategia splitteo: groupKfold, teniendo la misma justificacion que el binario.

Caso Binario

In [ ]:
test_original = pd.read_csv("../data/rendimiento_estudiantes_test.csv")
test = test_original.copy()

Convierto targets en valores numericos

In [ ]:
test['rendimiento_binario'] = test['rendimiento'].apply(lambda x: 0 if x == 'Insuficiente' else 1)
test['rendimiento_num'] = test['rendimiento'].map({'Insuficiente': 0, 'Regular': 1, 'Bueno': 2, 'Excelente': 3})

Paso semestres a valores numericos ordenados cronologicamente

In [ ]:
semestres_ordenados = sorted(test['semestre'].unique(), key=lambda x: (x.split('-')[0], x.split('-')[1]))
orden_semestres = {semestres_ordenados[i]: i for i in range(len(semestres_ordenados))}
test['semestre_num'] = test['semestre'].map(orden_semestres)


In [ ]:
test['escuela'] = test['escuela'].str.strip().str.upper()

Hago one hot de los colegios

In [ ]:
colegios = ['D', 'H', 'A', 'F', 'B', 'E', 'G', 'C']
for cole in colegios:
    test[f'es_{cole}'] = (test['escuela'] == cole).astype(int)

In [ ]:
test_target_binario = test.drop('rendimiento_num', axis=1)

Normalizo y agrego valores a los NaNs

In [ ]:
reemplazo_NaNs(dev_target_binario, test_target_binario, group_key, columnas_continuas)
dev_norm, test_norm, _ = normalizar(dev_target_binario, test_target_binario, columnas_continuas)

In [ ]:
X_dev = dev_norm[nombres_features].values
y_dev = dev_norm['rendimiento_binario'].values
X_test = test_norm[nombres_features].values
y_test_real = test_norm['rendimiento_binario'].values

Aplico SMOTE en todo el dev

In [ ]:
X_dev_bal, y_dev_bal = smote(X_dev, y_dev)

Entreno modelo final

In [ ]:
modelo_final_bin = LogisticRegression(X_dev_bal, y_dev_bal, nombres_features,L2=mejores_lambdas['group'], alpha=alfa, iters=iteraciones)
modelo_final_bin.fit()

In [ ]:
y_test_pred = modelo_final_bin.predecir_clase(X_test, umbral)
y_test_proba = modelo_final_bin.predecir_proba(X_test)

In [ ]:
acc = accuracy(y_test_pred, y_test_real, clase_positiva=clase_pos)
prec = precision(y_test_pred, y_test_real, clase_positiva=clase_pos)
rec = recall(y_test_pred, y_test_real, clase_positiva=clase_pos)
f1 = F1_score(y_test_pred, y_test_real, clase_positiva=clase_pos)
fpr, tpr, _ = curva_ROC(y_test_proba, y_test_real, clase_positiva=clase_pos)
auc_roc = AUC_ROC(fpr, tpr)
rec_pr, prec_pr, _ = curva_PR(y_test_proba, y_test_real, clase_positiva=clase_pos)
auc_pr = AUC_PR(rec_pr, prec_pr)

print(f'Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, AUC-ROC={auc_roc:.4f}, AUC-PR={auc_pr:.4f}')

Al evaluar el modelo en el test, se puede ver que mantiene un recall muy alto (0.97), es decir que detecta casi todos los estudiantes en riesgo. Igualmente, al tener una precision baja (0.3), se obtienen muchos falsos positivos. Como el objetivo del ministerio es encontrar los riesgosos, esto puede ser aceptable. El AUC-ROC (0.92), confirma que el modelo tiene buena capacidad de discriminacion.

Es importante utilizar rebalanceo para el modelo porque en el train set que tenemos hay muy pocos valores sobre la clase que nos importa, pero al mismo tiempo al tener que generar tantos valores (falsos) se corre el riesgo de cometer mas errores (que fue lo que sucedio).

La caida de precision respecto al ejercicio 4 (de 0.49 a 0.3), se debe a que en el test hay escuelas completamente nuevas, lo cual demuestra que es el escenario mas realista pero complejo.

Multiclase

In [ ]:
test_target_multiclase = test.drop('rendimiento_binario', axis=1)

In [ ]:
dev_multi_copy = dev_target_multiclase.copy()
test_multi_copy = test_target_multiclase.copy()

Normalizo y reemplazo NaNs

In [ ]:
reemplazo_NaNs(dev_multi_copy, test_multi_copy, group_key, columnas_continuas)
dev_multi_norm, test_multi_norm, _ = normalizar(dev_multi_copy, test_multi_copy, columnas_continuas)

In [ ]:
X_dev_multi = dev_multi_norm[nombres_features].values
y_dev_multi = dev_multi_norm['rendimiento_num'].values
X_test_multi = test_multi_norm[nombres_features].values
y_test_multi_real = test_multi_norm['rendimiento_num'].values

Entreno modelo final con LogisticRegressionMulti

In [ ]:
modelo_final_multi = LogisticRegressionMulticlase(X_dev_multi, y_dev_multi, nombres_features, L2=0, alfa=alfa, iters=iteraciones)
modelo_final_multi.fit()

Evaluo

In [ ]:
y_test_multi_pred = modelo_final_multi.predecir_clase(X_test_multi)
y_test_multi_proba = modelo_final_multi.predecir_proba(X_test_multi)

Metricas

In [ ]:
clases = [0, 1, 2, 3]
nombres_clases = ['Insuficiente', 'Regular', 'Bueno', 'Excelente']

for c, nombre in zip(clases, nombres_clases):
    p = precision(y_test_multi_pred, y_test_multi_real, clase_positiva=c)
    r = recall(y_test_multi_pred, y_test_multi_real, clase_positiva=c)
    f1 = F1_score(y_test_multi_pred, y_test_multi_real, clase_positiva=c)
    acc = accuracy(y_test_multi_pred, y_test_multi_real, clase_positiva=c)
    print(f'{nombre}: Accuracy={acc:.4f}, Precision={p:.4f}, Recall={r:.4f}, F1={f1:.4f}')

print(f'\nMatriz de confusion:')
print(matriz_confusion_multiclase(y_test_multi_pred, y_test_multi_real))

Hago Learning curve para Random Forest multiclase

Se entrena el modelo con distintas cantidades de datos y se va viendo como mejora el accuracy. Al graficar esto se ppuede estimar cuantos datos adicionales necesitamos para ganar un 1% de accuracy. Si la curva se aplana, significa que agregar mas datos no ayuda. Si tiene pendiente podemos extrapolar cuanto datos necesitamos.

In [ ]:
#se fija validation siempre con las mismas escuelas
_, val_fija = group_split(dev_target_multiclase, 6, rand_state=42)

escuelas_val = val_fija['escuela'].unique()
train_base = dev_target_multiclase[~dev_target_multiclase['escuela'].isin(escuelas_val)]

reemplazo_NaNs(train_base, val_fija, group_key, columnas_continuas)
_, val_fija_norm, _ = normalizar(train_base, val_fija, columnas_continuas)
X_val_fija = val_fija_norm[nombres_features].values
y_val_fija = val_fija_norm['rendimiento_num'].values

tamanios = [int(len(train_base) * f) for f in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]]
accuracies = []

val_fija_original = val_fija.copy()

for tam in tamanios:
    train_sample = train_base.sample(n=tam, random_state=42)
    val_fija_copia = val_fija_original.copy()
    
    reemplazo_NaNs(train_sample, val_fija_copia, group_key, columnas_continuas)
    train_norm, _, _ = normalizar(train_sample, val_fija_copia, columnas_continuas)
    
    X_tr = train_norm[nombres_features].values
    y_tr = train_norm['rendimiento_num'].values
    
    rf = RandomForest(X_tr, y_tr, nombres_features, n_arboles=50, max_profundidad=15, min_muestras_hoja=3)
    rf.fit()
    y_pred = rf.predecir_clase(X_val_fija)
    
    acc = (y_pred == y_val_fija).mean()
    accuracies.append(acc)
    print(f'n={tam}: accuracy={acc:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(tamanios, accuracies, marker='o')
plt.xlabel('Cantidad de datos de entrenamiento')
plt.ylabel('Accuracy')
plt.title('Learning Curve - Random Forest Multiclase')
plt.grid(True)
plt.show()

acc_actual = accuracies[-1]
acc_objetivo = acc_actual + 0.01
print(f'Accuracy actual con todos los datos: {acc_actual:.4f}')
print(f'Accuracy objetivo (+1%): {acc_objetivo:.4f}')

Al aumentar la cantidad de datos, el accuracy mejora al principio, pero luego deja de mejorar de forma clara, empezando a variar al rededor de un valor similar. Esto demuestra que agregar mas datos ya no produce una mejor muy significativa en el modelo.

El accuracy objetivo es 0.5307, pero en la curva no se puede ver que el modelo llegue a un valor cercano a eso, ni utilizando datos disponibles. No se podria estimar cuantos datos extra serian necesarios, ya que solo agregar datos no parece ser suficiente para lograr la mejora. Se deberia mejorar el modelo o las variables utilizadas.